# Universal plots from the 925 semantic trees

Plotting cells adapted from `code/tree_universal.ipynb` (cells 4 and 9).
The original six-bin histogram calculations and plotting settings are retained.
The input loader reads the bundled archive, uses `N=len(saved tokens)` as in
`tree_analysis`, and excludes each tree's deepest level as in `collect_level_data`.
The only presentation adjustments are tight layout and a valid positive lower
bound for the logarithmic size axis. Run all cells from the repository or its
`notebooks` directory. No model, tokenizer, or external dataset is needed.

Original notebook SHA-256: `d5515f9b7b2191979bfef15034eb8d2c09c0d190931a3d8b6d2eb823931edd7a`


In [ ]:
from pathlib import Path
import sys
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.ticker import LogLocator, NullFormatter

ROOT = Path.cwd()
if not (ROOT/'data/trees_cut30.tar.gz').is_file():
    ROOT = ROOT.parent
sys.path.insert(0,str(ROOT))
from analysis.common import iter_trees, style
from analysis.figure5 import notebook_samples
style()
trees = list(iter_trees(ROOT/'data/trees_cut30.tar.gz'))
assert len(trees)==925
# Original plotting cells concatenate per-story arrays. Pooling in the loader
# gives the same samples; the singleton list retains that input interface.
s_data = {L:[values] for L,values in notebook_samples(trees).items()}
L_pts = np.arange(2,12)
OUT = ROOT/'figures'
OUT.mkdir(parents=True,exist_ok=True)
print(f'{len(trees)} trees; normalization: n / saved token count')


## Standardized collapse: original cell 4

In [ ]:
n_bins = 6
L_pts = np.arange(2, 12, 1)
colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(L_pts)))

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
custom_cmap = ListedColormap(colors)

# Set up the colorbar with the custom colormap
norm = BoundaryNorm(boundaries=np.arange(-0.5, len(L_pts)+0.5), ncolors=custom_cmap.N)

# Create a ScalarMappable and use our custom colormap and norm
sm = plt.cm.ScalarMappable(cmap=custom_cmap, norm=norm)
sm.set_array([])

# Adding the colorbar to the figure with customized ticks and labels
cbar = plt.colorbar(sm, ax=ax, ticks=np.arange(0, len(L_pts), 1))
cbar.set_label('Level' + r'$(L)$', rotation=90)
cbar.set_ticklabels([int(np.rint(L_pts[i])) for i in range(0, len(L_pts), 1)])


for idx, L in enumerate(L_pts):
    lists_L = s_data.get(L, [])
    if not lists_L or not any(len(v) for v in lists_L):
        continue

    s_vals = np.concatenate([np.asarray(v) for v in lists_L if len(v)])
    s_vals[s_vals == 0] = np.nan
    z_vals = np.log(s_vals)
    mu_emp  = np.nanmean(z_vals)
    sigma_emp = np.nanstd(z_vals)
    x_vals = (z_vals - mu_emp) / sigma_emp

    bins     = np.linspace(np.nanmin(x_vals), np.nanmax(x_vals), n_bins + 1)
    h_emp, _ = np.histogram(x_vals, bins=bins, density=True)
    x_emp    = (bins[:-1] + bins[1:]) / 2

    ax.plot(x_emp, h_emp, 'o', ms=5, color=colors[idx], alpha=1)

ax.plot(np.linspace(-5, 5, 100), scipy.stats.norm.pdf(np.linspace(-5, 5, 100)),
        'k--', lw=4, label='$\mathcal{N}(0,1)$')  # master curve

ax.set_xlabel(r'$(\ln (s) - \hat{\mu}_L )/\hat{\sigma}_L$')
ax.set_ylabel(r'$\hat{\sigma}_L s \hat{f}_{L}(s)$')
ax.set_ylim([0, 0.725])
ax.set_xlim([-5, 5])
ax.legend(loc='upper right')
fig.tight_layout()
fig.savefig(OUT/'figure5d_collapse_925.pdf')

## Size density: original cell 9

In [ ]:
n_bins = 6

def plot_asymp_dist_log(ax, hist, bins, color, i):
    log_hist = np.log(hist)
    log_hist[np.isneginf(log_hist)] = np.nan
    valid = ~np.isnan(log_hist)
    ax.loglog(bins[valid], hist[valid], 'o-', linewidth=3, color=color, alpha=0.8)

colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(L_pts)))

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
custom_cmap = ListedColormap(colors)

# Set up the colorbar with the custom colormap
norm = BoundaryNorm(boundaries=np.arange(-0.5, len(L_pts)+0.5, 1), ncolors=custom_cmap.N)

# Create a ScalarMappable and use our custom colormap and norm
sm = plt.cm.ScalarMappable(cmap=custom_cmap, norm=norm)
sm.set_array([])

# Adding the colorbar to the figure with customized ticks and labels
cbar = plt.colorbar(sm, ax=ax, ticks=np.arange(0, len(L_pts)))
cbar.set_label('Level' + r'$(L)$', rotation=90)
cbar.set_ticklabels([int(np.rint(L_pts[i])) for i in range(0, len(L_pts))])


for idx, L in enumerate(L_pts):
    # pooled empirical density f_L(s)
    lists_L = s_data.get(L, [])
    # if nothing at this level (or all empties), skip
    if not lists_L or not any(len(v) for v in lists_L):
        ax.set_title(f"Level {L} - no data"); continue

    # concatenate only the non-empty chunks
    s_vals = np.concatenate([np.asarray(v) for v in lists_L if len(v)])

    s_vals[s_vals == 0] = np.nan
    x_vals = s_vals

    # empirical range + histogram (density=True -> PDF)
    bins     = np.linspace(np.nanmin(x_vals), np.nanmax(x_vals), n_bins + 1)
    h_emp, _ = np.histogram(x_vals, bins=bins, density=True)
    x_emp    = (bins[:-1] + bins[1:]) / 2    # bin centres

    plot_asymp_dist_log(ax, h_emp, x_emp, colors[idx], idx)

ax.set_xlabel(r'$s$')
ax.set_ylabel(r'$\hat{f}_{L}(s)$')
ax.set_ylim([10**(-4), 10**(4)])
ax.set_xlim(right=1)  # zero is outside the logarithmic domain

# Set minor ticks to be dense between the major ticks
subs = np.arange(2, 10) * 0.1  # Subdivisions between powers of 10
ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=subs, numticks=100))
ax.yaxis.set_minor_formatter(NullFormatter())  # Hide minor tick labels
fig.tight_layout()
fig.savefig(OUT/'figure5c_density_925.pdf')